Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
from neuralforecast import NeuralForecast
from neuralforecast.models import Informer, Autoformer,  PatchTST, TFT
from neuralforecast.losses.numpy import mse
from neuralforecast.losses.pytorch import MSE
np.random.seed(42)



In [2]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
else:
    print("No GPU available. Training will run on CPU.")

GPU: NVIDIA GeForce RTX 2050 is available.


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [4]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(43800, 6)

In [10]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


In [11]:
datosNormalizados['ds'] = pd.to_datetime(datosNormalizados.index)
datosNormalizados["unique_id"] = 1


In [12]:

datosNormalizados.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,ds,unique_id
date,,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,2010-01-02 00:00:00,1
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,2010-01-02 01:00:00,1
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,2010-01-02 02:00:00,1
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,2010-01-02 03:00:00,1
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,2010-01-02 04:00:00,1


Espacio de búsqueda

In [14]:
'''        

"input_size_multiplier": [1, 2, 3, 4, 5],
        "h": None,
        "n_head": tune.choice([4, 8]),
        "learning_rate": tune.loguniform(1e-4, 1e-1),
        "scaler_type": identity # tambien usar local_scaler_type='identity' en neural_forecast()
        "max_steps": tune.choice([500, 1000, 2000]),
        "batch_size": 1
        "windows_batch_size": tune.choice([128, 256, 512, 1024]),
        "loss": MSE,
        "random_seed": tune.randint() #42,
        inference_windows_batch_size = windows_batch

'''

'        \n\n"input_size_multiplier": [1, 2, 3, 4, 5],\n        "h": None,\n        "n_head": tune.choice([4, 8]),\n        "learning_rate": tune.loguniform(1e-4, 1e-1),\n        "scaler_type": identity # tambien usar local_scaler_type=\'identity\' en neural_forecast()\n        "max_steps": tune.choice([500, 1000, 2000]),\n        "batch_size": 1\n        "windows_batch_size": tune.choice([128, 256, 512, 1024]),\n        "loss": MSE,\n        "random_seed": tune.randint() #42,\n        inference_windows_batch_size = windows_batch\n\n'

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 24
pasados  = 12

In [14]:
datosNormalizados['y'] = datosNormalizados['pollution'].shift(- futuros)

In [15]:
datosX = datosNormalizados.dropna()

datosX.head(40)

,pollution,dew,temp,press,wnd_dir,wnd_spd,ds,unique_id,y
date,,,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,2010-01-02 00:00:00,1,-0.110234
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,2010-01-02 01:00:00,1,-0.406482
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,2010-01-02 02:00:00,1,-0.384538
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,2010-01-02 03:00:00,1,-0.494260
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,2010-01-02 04:00:00,1,-0.384538
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017,2010-01-02 05:00:00,1,-0.187039
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876,2010-01-02 06:00:00,1,-0.099262
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735,2010-01-02 07:00:00,1,-0.154122
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594,2010-01-02 08:00:00,1,-0.198011


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30643, 9)
Las dimensiones de testX son:  (8799, 9)
Las dimensiones de valX son:  (4334, 9)


In [17]:
trainX = pd.concat([trainX, testX], axis=0)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de valX son: ", valX.shape)

Las dimensiones de trainX son:  (39442, 9)
Las dimensiones de valX son:  (4334, 9)


Se crean métricas para medir desempeño

In [20]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [21]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [34]:
def objective(params):

  steps_per_epoch = int(len(trainX) // int(params["windows_batch"]))
  val_check_steps = steps_per_epoch  # Validate once per epoch
  early_stop_patience_steps = 15 * steps_per_epoch  # Stop if no improvement for 15 epochs


  max_steps = steps_per_epoch * params["epochs"]
  print("##########")
  print("early_stopping_patiente: ", early_stop_patience_steps)
  print("max_steps:", max_steps)
  print("##########")

  informerModel =  TFT(h=futuros,
                  input_size=pasados,
                  hidden_size = int(params["hidden_size"]),
                  #conv_hidden_size = int(params["conv_hidden_size"]),
                  #n_head= int(params["n_head"]),
                  loss=MSE(),
                  #scaler_type='identity',
                  learning_rate=float(params["learning_rate"]),
                  #activation= params["activation"],
                  #factor= int(params["factor"]),
                  #distil= bool(params["distil"]),
                  #dropout=float(params["dropout"]),
                  #encoder_layers= int(params["encoder_layers"]),
                  #decoder_layers= int(params["decoder_layers"]),
                  #batch_size= 1,
                  #windows_batch_size= int(params["windows_batch"]),
                  val_check_steps=val_check_steps,
                  early_stop_patience_steps=early_stop_patience_steps,
                  max_steps=max_steps,
                  exclude_insample_y=True,
                  inference_windows_batch_size= int(params["windows_batch"]),
                  random_seed=42,
                  hist_exog_list = ["pollution",	"dew",	"temp",	"press",	"wnd_dir",	"wnd_spd"]
                  )




  nf = NeuralForecast(
      models=[informerModel],
      freq='H', local_scaler_type=None)

  nf.fit(df=trainX, val_size=len(testX), verbose= True)

  forecasts = nf.predict(df=valX)

  loss = mse(valX['y'].values, forecasts['InformerModel'].values)
  

  return {'loss': loss, 'status': STATUS_OK}

In [19]:
space = {
    'hidden_size': hp.choice('hidden_size',[64, 128, 256]),
    'n_head': hp.quniform('n_head', 4, 8, 1),
    'encoder_layers': hp.quniform('encoder_layers', 1, 4, 1),
    'decoder_layers': hp.quniform('decoder_layers', 1, 4, 1),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'windows_batch': hp.choice('windows_batch',[2 ** i for i in range(3, 9)]),
    'activation': hp.choice('activation', ['gelu', 'relu']),
    'factor': hp.quniform('factor', 3, 7, 2),
    "conv_hidden_size": hp.choice('conv_hidden_size', [64, 128, 256]),
    "distil": hp.choice('distil', [True, False]),
}
'''
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}
'''

"\n    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU\n    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU\n    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),\n    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización\n    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje\n    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento\n    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])\n}\n"

In [35]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))


                                                   ##########

                                                   early_stopping_patiente: 

                                                   73950

                                                   max_steps:

                                                   157760

                                                   ##########


Seed set to 42
job exception: Trainer.__init__() got an unexpected keyword argument 'num_workers'



  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]


TypeError: Trainer.__init__() got an unexpected keyword argument 'num_workers'

In [ ]:
print(best)